In [274]:
import os

import numpy as np
import pandas as pd
import glob
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity

# rawdata에서 피험자별 seedword 순서 정보 추출

In [275]:
data_dir = os.getcwd() + '/data/' # data위치 지정
datasets = glob.glob( data_dir + 'raw/' + 'rawdata*.xls' )


In [276]:
seeds_of_subjects = []

# rawdata_study1.xls 피험자 137명
for sheet_index in range(1, 138): 
    seeds_of_each_subject = []
    
    # 시트 차례로 불러오기
    tbl_data = pd.read_excel(datasets[0], sheet_name=f'Sheet{sheet_index}', header=None)
    # row별로 dict 생성
    tbl_data_dict = tbl_data.to_dict('records') # row별로 dict 생성

    for i, value in enumerate(tbl_data_dict):
        if value[4].replace("'", "") == 'seedword':
            seeds_of_each_subject.append(value[10].replace("'", ""))
    seeds_of_subjects.append(seeds_of_each_subject)


In [277]:
# rawdata_study2.xls 피험자 213명
for sheet_index in range(1, 214): 
    seeds_of_each_subject = []
    
    # 시트 차례로 불러오기
    tbl_data = pd.read_excel(datasets[1], sheet_name=f'Sheet{sheet_index}', header=None)
    # row별로 dict 생성
    tbl_data_dict = tbl_data.to_dict('records') # row별로 dict 생성

    for i, value in enumerate(tbl_data_dict):
        if value[4].replace("'", "") == 'seedword':
            seeds_of_each_subject.append(value[10].replace("'", ""))
    seeds_of_subjects.append(seeds_of_each_subject)


In [278]:
print(f'''
seeds_of_subjects는 rawdata엑셀 파일에서, 각 피험자별 seed단어 순서들을 추출해서 저장해둔 배열입니다. \n
{len(seeds_of_subjects)} \n
       ''')
print(f'예시: {seeds_of_subjects[0:3]}')


seeds_of_subjects는 rawdata엑셀 파일에서, 각 피험자별 seed단어 순서들을 추출해서 저장해둔 배열입니다. 

350 

       
예시: [['tear', 'family', 'mirror', 'abuse'], ['tear', 'abuse', 'mirror', 'family'], ['family', 'mirror', 'tear', 'abuse']]


# web1, web2 npy데이터 열기

In [279]:
raw_data_dir = os.getcwd() + '/data/raw/' # data위치 지정

# npy 파일 열기
web1_vectors = np.load(raw_data_dir + 'web1-sess1.npy')
web2_vectors = np.load(raw_data_dir + 'web2-sess1.npy')

# vstack: vertical stack을 해주는 함수. 작성 순서대로(왼->오) 순서가 보장된 채로 하나로 합쳐주는 역할
all_data_vectors = np.vstack((web1_vectors, web2_vectors))
# 배열 shape 숫자들을 변수에 저장
n_subject, n_seed, n_response_words, n_dim_of_vector = all_data_vectors.shape

print(f'피험자 숫자: {n_subject}\nseed개수: {n_seed}\nseed당 응답 단어 개수: {n_response_words}\n벡터의 차원: {n_dim_of_vector}')

피험자 숫자: 350
seed개수: 4
seed당 응답 단어 개수: 40
벡터의 차원: 90


In [280]:
total = [] # 전체 벡터데이터 저장할 배열

for i_subject in range(n_subject): ##### 각 피험자의
    response_words_of_each_subject = {
        'subject': i_subject+1, # ex) subject: 1
    }

    for i_seed in range(n_seed): ##### 각 seed단어의
        # rawdata에서 뽑아온 seedword정보에서 해당 피험자 찾아서 seedword가져오기
        seedword = seeds_of_subjects[i_subject][i_seed] 

        for i_word in range(n_response_words): ##### 각 응답단어의 벡터 (90차원)
            # response_words_of_each_subject 딕셔너리에 ex. abuse1_vec 컬럼명을 찾아서 벡터값 저장
            response_words_of_each_subject[f'{seedword}{i_word+1}_vec'] = all_data_vectors[i_subject][i_seed][i_word] # ex) abuse1_vec: 90차원벡터값

    total.append(response_words_of_each_subject)

In [281]:
print(f'number of subject: {len(total)}')
print(f'data of subject1: \n {total[0]}')

number of subject: 350
data of subject1: 
 {'subject': 1, 'tear1_vec': array([-2.79974675,  1.00251234, -1.22222555, -0.9890914 ,  1.18628621,
        4.88525486, -1.98106539, -2.03732109, -2.85267663,  2.81611705,
        2.73404026,  7.16165304, -4.44087505,  2.02881861, -2.95463133,
        0.83100581, -0.9974094 , -0.16849989, -1.25637519,  0.47775674,
       -0.6926859 , -2.266855  , -2.72205043,  0.22537686,  0.89014977,
       -1.19019365, -2.10014009,  3.52187634, -1.76154828, -1.11329436,
       -3.43915224, -0.31275928,  3.28120947,  4.59352732, -1.05789196,
       -1.12197518, -2.38847589,  0.91673404, -4.85794258, -0.5881058 ,
       -1.3501513 , -0.41639099, -3.60945988, -3.546947  ,  1.4391768 ,
       -3.10977888,  2.88338351,  5.93057013,  1.29082584,  1.84497523,
       -2.08638811,  1.33774412, -3.15434217,  2.02142429,  3.19965053,
       -0.30009282, -3.69844437,  1.13195777,  0.06201917, -0.02972814,
       -4.44109058,  1.67239642, -3.48865008,  2.62012172,  1.377

In [282]:
all_data_df = pd.DataFrame(total)
all_data_df

,subject,tear1_vec,tear2_vec,tear3_vec,tear4_vec,tear5_vec,tear6_vec,tear7_vec,tear8_vec,tear9_vec,...,abuse31_vec,abuse32_vec,abuse33_vec,abuse34_vec,abuse35_vec,abuse36_vec,abuse37_vec,abuse38_vec,abuse39_vec,abuse40_vec
0,1,"[-2.7997467517852783, 1.0025123357772827, -1.2...","[-1.5049004554748535, 1.5659693479537964, 1.50...","[0.0731828510761261, -0.004675315693020821, 0....","[-0.03984304144978523, -0.03075242042541504, 0...","[-0.00021356847719289362, 5.758549213409424, -...","[-0.05162226781249046, -0.08953678607940674, -...","[-1.0766730308532715, 0.10434307157993317, 5.6...","[2.424929141998291, 3.6071736812591553, -3.387...","[-5.876298666000366, 0.33746337890625, -2.5496...",...,"[0.3068299889564514, -5.440547704696655, -0.15...","[-3.669671967625618, -5.602187544107437, 0.863...","[0.4952196478843689, 2.1568381786346436, 1.788...","[2.353285074234009, 3.7313449382781982, -0.480...","[1.4057475328445435, 2.7502248287200928, 3.187...","[3.951447606086731, 1.5583451390266418, -0.109...","[2.310206413269043, 1.7991424798965454, 2.6263...","[5.4977288246154785, -1.2148534059524536, 4.73...","[-5.164267539978027, -0.7331278324127197, 1.05...","[-0.35834380984306335, 1.018376350402832, -0.7..."
1,2,"[-2.7997467517852783, 1.0025123357772827, -1.2...","[0.45708733797073364, -0.6215798258781433, -0....","[-5.959320068359375, 6.186459600925446, -2.712...","[-2.988236904144287, 0.7769572138786316, 0.323...","[0.5567496418952942, -2.2172653675079346, 2.29...","[1.8707859516143799, -1.1655787229537964, 1.85...","[-1.9872565269470215, -9.86076545715332, 2.282...","[-1.0783004760742188, 1.3618557453155518, -2.3...","[-0.757293164730072, -0.211566761136055, -0.84...",...,"[-2.8006346225738525, -3.958317756652832, -1.5...","[-2.8006346225738525, -3.958317756652832, -1.5...","[-1.9371036291122437, -3.3271970748901367, -3....","[1.769850254058838, 3.052565097808838, -3.1117...","[0.18298132717609406, 1.612946629524231, -3.05...","[0.1730307936668396, 0.038090191781520844, 0.1...","[2.647094964981079, 0.6997803449630737, -1.559...","[-0.17232778668403625, -5.012906551361084, -6....","[-2.4440691471099854, -0.8925318717956543, -0....","[1.687965750694275, -0.9570441842079163, 1.968..."
2,3,"[-0.14277277886867523, 0.17866025865077972, -1...","[-0.05008912459015846, 4.079308986663818, -5.3...","[1.6711207628250122, -4.349452972412109, -2.34...","[-1.169617772102356, 0.7114548683166504, -2.06...","[1.765024185180664, -1.3058565855026245, -5.42...","[-0.1085025742650032, 0.5090253949165344, -2.0...","[2.6600513458251953, 1.1037150621414185, -1.56...","[2.0577778816223145, 2.3813185691833496, 3.318...","[-5.1105570793151855, 0.6133217215538025, -2.9...",...,"[-1.3444898128509521, -1.9484931230545044, -3....","[-0.5626689195632935, 1.0186009407043457, 1.79...","[-0.004942434839904308, -0.00423665763810277, ...","[7.317351341247559, 0.19146670401096344, -0.83...","[-0.7530180811882019, -0.4503012001514435, 0.6...","[3.7946832180023193, 1.2103089094161987, -0.39...","[-6.381642818450928, -0.25714749097824097, 2.7...","[-0.8012444376945496, -1.6405866146087646, 1.0...","[-0.679341197013855, -0.5665156245231628, -0.2...","[-1.0201469659805298, -2.0769898891448975, 0.1..."
3,4,"[-4.05358362197876, -2.1990063190460205, -2.74...","[0.006588673684746027, 0.11960766464471817, -0...","[-3.8310554027557373, 0.4362701177597046, -1.6...","[-4.372428119182587, 7.190983533859253, 6.6034...","[1.6293264627456665, 3.593465566635132, -4.204...","[-1.5804722309112549, -1.4650334119796753, -1....","[-0.4196225702762604, -3.142327070236206, -5.8...","[0.22102010250091553, -3.767268657684326, -6.0...","[0.1270749568939209, -1.7401845455169678, -4.4...",...,"[4.43143367767334, 2.9310171604156494, -0.9024...","[-4.698283672332764, 0.11811280995607376, 6.05...","[-0.8674511909484863, -0.18062961101531982, 0....","[-4.729660511016846, -1.2575398683547974, 5.03...","[-1.4293122291564941, 0.7171301245689392, 1.09...","[-5.283805847167969, -4.199435710906982, 1.504...","

# similarity - coherence

유사도 방법론: https://wikidocs.net/24654

코드 구성은 기본적으로 2_get_similarity_and_coherence.ipynb와 동일합니다.

In [283]:
# loading the word2vec model
word2vec_model = KeyedVectors.load_word2vec_format('pretrained/GoogleNews-vectors-negative300.bin', binary=True)
# print(word2vec_model['king']) # 모델이 잘 로드되었는지 확인

In [284]:
seed_words = ['abuse', 'tear', 'mirror', 'family'] # 학대, 눈물, 거울, 가족
target_words = ['money', 'friend', 'relationships', 'family']

In [285]:
# coherence 컬럼 미리 생성(빈 값)
for seed_word in seed_words:
    for target_word in target_words:
        column_name = f'coherence_{seed_word}_{target_word}'
        all_data_df = all_data_df.assign(**{column_name: None})

all_data_df.iloc[0:3]


,subject,tear1_vec,tear2_vec,tear3_vec,tear4_vec,tear5_vec,tear6_vec,tear7_vec,tear8_vec,tear9_vec,...,coherence_tear_relationships,coherence_tear_family,coherence_mirror_money,coherence_mirror_friend,coherence_mirror_relationships,coherence_mirror_family,coherence_family_money,coherence_family_friend,coherence_family_relationships,coherence_family_family
0,1,"[-2.7997467517852783, 1.0025123357772827, -1.2...","[-1.5049004554748535, 1.5659693479537964, 1.50...","[0.0731828510761261, -0.004675315693020821, 0....","[-0.03984304144978523, -0.03075242042541504, 0...","[-0.00021356847719289362, 5.758549213409424, -...","[-0.05162226781249046, -0.08953678607940674, -...","[-1.0766730308532715, 0.10434307157993317, 5.6...","[2.424929141998291, 3.6071736812591553, -3.387...","[-5.876298666000366, 0.33746337890625, -2.5496...",...,None,None,None,None,None,None,None,None,None,None
1,2,"[-2.7997467517852783, 1.0025123357772827, -1.2...","[0.45708733797073364, -0.6215798258781433, -0....","[-5.959320068359375, 6.186459600925446, -2.712...","[-2.988236904144287, 0.7769572138786316, 0.323...","[0.5567496418952942, -2.2172653675079346, 2.29...","[1.8707859516143799, -1.1655787229537964, 1.85...","[-1.9872565269470215, -9.86076545715332, 2.282...","[-1.0783004760742188, 1.3618557453155518, -2.3...","[-0.757293164730072, -0.211566761136055, -0.84...",...,None,None,None,None,None,None,None,None,None,None
2,3,"[-0.14277277886867523, 0.17866025865077972, -1...","[-0.05008912459015846, 4.079308986663818, -5.3...","[1.6711207628250122, -4.349452972412109, -2.34...","[-1.169617772102356, 0.7114548683166504, -2.06...","[1.765024185180664, -1.3058565855026245, -5.42...","[-0.1085025742650032, 0.5090253949165344, -2.0...","[2.6600513458251953, 1.1037150621414185, -1.56...","[2.0577778816223145, 2.3813185691833496, 3.318...","[-5.1105570793151855, 0.6133217215538025, -2.9...",...,None,None,None,None,None,None,None,None,None,None


In [286]:
# 원래의 300차원 워드 벡터 생성
money_vec = word2vec_model['money'][:90]
friend_vec = word2vec_model['friend'][:90]
relationships_vec = word2vec_model['relationships'][:90]
family_vec = word2vec_model['family'][:90]

for seed_word in seed_words:
    coherences_money_per_sub = []
    coherences_friend_per_sub = []
    coherences_relationships_per_sub = []
    coherences_family_per_sub = []
    
    word_columns = [seed_word + str(i) for i in range(1, n_response_words + 1)]

    for i_subject in range( n_subject): # 1 ~ 350번 피험자
        each_seed_similarities_money = []  
        each_seed_similarities_friend = []  
        each_seed_similarities_relationships = []
        each_seed_similarities_family = []

        for i, column in enumerate(word_columns): # 각각 seed1~40
            try:
                similarity_between_money = cosine_similarity(all_data_df.iloc[i_subject][f'{column}_vec'].reshape(1, -1), money_vec.reshape(1, -1))[0][0]
                similarity_between_friend = cosine_similarity(all_data_df.iloc[i_subject][f'{column}_vec'].reshape(1, -1), friend_vec.reshape(1, -1))[0][0]
                similarity_between_relationships = cosine_similarity(all_data_df.iloc[i_subject][f'{column}_vec'].reshape(1, -1), relationships_vec.reshape(1, -1))[0][0]
                similarity_between_family = cosine_similarity(all_data_df.iloc[i_subject][f'{column}_vec'].reshape(1, -1), family_vec.reshape(1, -1))[0][0]

                each_seed_similarities_money.append(similarity_between_money) # 40개
                each_seed_similarities_friend.append(similarity_between_friend) # 40개
                each_seed_similarities_relationships.append(similarity_between_relationships) # 40개
                each_seed_similarities_family.append(similarity_between_family) # 40개
            except Exception:
                continue  
        # 해당 피험자가 하나의 seed에 답한 40개의 응답단어의 유사도의 coherence값
        money_coherence_of_seed_per_sub = np.mean(each_seed_similarities_money) 
        friend_coherence_of_seed_per_sub = np.mean(each_seed_similarities_friend)
        relationships_coherence_of_seed_per_sub = np.mean(each_seed_similarities_relationships) 
        family_coherence_of_seed_per_sub = np.mean(each_seed_similarities_family)

        # 해당 피험자에 대한, 그 seed단어 각각(4개)에 대한 coherence값
        # 피험자마자 16개값( abuse,tear,mirror,family - money,friend,relationships,family 조합)
        coherences_money_per_sub.append(money_coherence_of_seed_per_sub)
        coherences_friend_per_sub.append(friend_coherence_of_seed_per_sub)
        coherences_relationships_per_sub.append(relationships_coherence_of_seed_per_sub)
        coherences_family_per_sub.append(family_coherence_of_seed_per_sub)

        # Append the coherence values to the data frame for the current subject
        all_data_df.at[i_subject, f'coherence_{seed_word}_money'] = money_coherence_of_seed_per_sub
        all_data_df.at[i_subject, f'coherence_{seed_word}_friend'] = friend_coherence_of_seed_per_sub
        all_data_df.at[i_subject, f'coherence_{seed_word}_relationships'] = relationships_coherence_of_seed_per_sub
        all_data_df.at[i_subject, f'coherence_{seed_word}_family'] = family_coherence_of_seed_per_sub



In [305]:
drop_columns = all_data_df.columns[1:161] # vector 컬럼들 드롭
all_data_df = all_data_df.drop(drop_columns, axis='columns')
all_data_df[0:3]

,subject,coherence_abuse_money,coherence_abuse_friend,coherence_abuse_relationships,coherence_abuse_family,coherence_tear_money,coherence_tear_friend,coherence_tear_relationships,coherence_tear_family,coherence_mirror_money,coherence_mirror_friend,coherence_mirror_relationships,coherence_mirror_family,coherence_family_money,coherence_family_friend,coherence_family_relationships,coherence_family_family
0,1,-0.0339,-0.070248,0.033578,0.038118,-0.015877,-0.09457,0.016956,0.040386,-0.03468,-0.056288,0.010468,0.011554,-0.014061,-0.042279,0.048968,0.034762
1,2,-0.041405,-0.053929,-0.005651,-0.014738,-0.033688,-0.09907,0.004897,-0.048921,0.006353,-0.053232,-0.014116,-0.000326,-0.026695,-0.065986,0.029354,-0.040201
2,3,-0.062901,-0.062581,0.053195,-0.004476,-0.004859,-0.04779,0.009106,0.049194,-0.053489,-0.067074,0.038215,0.040049,-0.042503,-0.085476,-0.011849,-0.007311


In [307]:
all_data_df.to_csv(data_dir + 'processed/similarity_coherence_data_90.csv', index=None)